In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

#leyendo base de datos
df = pd.read_csv('Book_bomberos_2015_a_sep_de_2025_ok_pa.csv',sep=";")

regiones_2 = ["anho","Región de Arica","Región de Tarapacá","Región de Antofagasta","Región de Atacama","Región de Coquimbo","Región de Valparaíso","Región Metropolitana","Región del Libertador Bernardo O'higgins","Región del Maule","Región del Ñuble","Región del Bío Bío","Región de la Araucanía","Región de los Ríos","Región de los Lagos","Región de Aysén del General Carlos Ibáñez del Campo","Región de Magallanes y de la Antártica Chilena"]
df = pd.read_csv('Book_bomberos_2015_a_sep_de_2025_ok_pa.csv',sep=";")
meses_a_numero = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4,
    "mayo": 5, "junio": 6, "julio": 7, "agosto": 8,
    "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12
}

df["mes_num"] = df["mes"].str.lower().map(meses_a_numero)

df_totales_anuales = df.groupby("anho").sum(numeric_only=True).reset_index() 

df.head()

,Unnamed: 0,tipo de incendio,año,mes,anho,Región de Arica,Región de Tarapacá,Región de Antofagasta,Región de Atacama,Región de Coquimbo,...,Región del Maule,Región del Ñuble,Región del Bío Bío,Región de la Araucanía,Región de los Ríos,Región de los Lagos,Región de Aysén del General Carlos Ibáñez del Campo,Región de Magallanes y de la Antártica Chilena,Chile,mes_num
0,0,Estructural 10-0,2015,enero,1,9,23,27,10,16,...,82,30,99,88,42,99,7,20,899,1
1,1,Vehículos 10-1,2015,enero,1,3,6,5,4,12,...,21,10,18,19,9,17,1,4,341,1
2,2,Pastizales y/o Basura 10-2,2015,enero,1,18,24,47,59,44,...,695,263,846,756,247,281,5,18,5419,1
3,3,Rescate de Emergencia 10-3,2015,enero,1,4,12,15,5,11,...,59,20,91,45,33,50,7,2,678,1
4,4,Rescate Vehicular 10-4,2015,enero,1,14,35,33,54,42,...,106,52,157,105,46,76,7,9,1330,1


In [2]:
import os
import sys
from contextlib import contextmanager
from pycaret.regression import *

# 1. Definimos una función para silenciar todo
@contextmanager
def suppress_output():
    with open(os.devnull, 'w') as fnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        try:
            sys.stdout = fnull
            sys.stderr = fnull
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

# Tu preparación de datos
a = 5
# ... (tu código de carga de datos y groupby) ...
# Suponiendo que df_totales_anuales ya está creado
data_py = df_totales_anuales[["año", "mes_num", "Chile"]].copy()

# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Chile", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Chile
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,982.1466,1795562.5021,1313.9266,0.7069,0.1167,0.0859,0.0620
et,Extra Trees Regressor,1047.6652,2080935.8157,1403.1738,0.6687,0.1251,0.0923,0.1280
xgboost,Extreme Gradient Boosting,1087.6367,2105907.0213,1427.7906,0.6571,0.1268,0.0954,0.3520
rf,Random Forest Regressor,1048.2481,2197373.2171,1450.0712,0.6396,0.1287,0.0928,0.1620
knn,K Neighbors Regressor,1269.5881,3065655.2526,1729.0852,0.4791,0.1497,0.1124,0.0460
ada,AdaBoost Regressor,1349.6622,3201746.6462,1779.6517,0.4509,0.1558,0.1198,0.0700
lightgbm,Light Gradient Boosting Machine,1393.4499,3514376.4701,1850.0553,0.4085,0.1608,0.1226,0.1900
dt,Decision Tree Regressor,1528.9667,4094895.5343,2007.8834,0.3114,0.1689,0.1298,0.0180
llar,Lasso Least Angle Regression,1623.2136,4670514.4196,2147.8074,0.1917,0.1853,0.1438,0.0180
en,Elastic Net,1623.2155,4670442.5464,2147.7956,0.1917,0.1853,0.1438,0.0140


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,978.2141,1754846.4469,1324.7062,0.7972,0.1096,0.0830


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,1400.0635,2643754.5751,1625.9627,0.6980,0.1374,0.1163
1,933.6213,2217958.2250,1489.2811,0.5803,0.1762,0.1046
2,791.2670,985238.6028,992.5919,0.7218,0.0878,0.0727
3,1072.4560,1775787.6024,1332.5868,0.6977,0.1034,0.0828
4,1277.1290,2301779.4935,1517.1617,0.6887,0.1149,0.0979
Mean,1094.9074,1984903.6998,1391.5168,0.6773,0.1239,0.0949
Std,221.3774,571291.3450,220.4192,0.0497,0.0307,0.0155


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,978.2141,1754846.4469,1324.7062,0.7972,0.1096,0.0830


¡Proceso terminado!
GradientBoostingRegressor(random_state=123)


La estrategia de validación a 5 folds ha resuelto definitivamente los problemas de inestabilidad estacional, confirmando al Gradient Boosting Regressor (GBR) como el modelo óptimo para la proyección nacional ("Chile"). El algoritmo demuestra ahora una solidez estructural impecable: todos los folds de validación arrojaron métricas positivas y consistentes ($R^2$ entre 0.58 y 0.72), eliminando por completo las divergencias negativas previas. En el conjunto de prueba final, el modelo alcanzó un $R^2$ de 0.80 junto con un MAPE del 8.30%, lo que implica un margen de error operativo mínimo y altamente competitivo. Estos resultados validan que, al garantizar ventanas temporales de entrenamiento más amplias, el modelo ha logrado capturar correctamente tanto la tendencia macro como los ciclos estacionales de los siniestros, constituyendo una herramienta robusta para la toma de decisiones.

In [3]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num", "Región de Arica"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Arica", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Arica")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Arica
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,10.2407,166.1260,12.7660,0.4946,0.1908,0.1565,0.1320
ada,AdaBoost Regressor,10.4188,164.2532,12.7703,0.4914,0.1909,0.1605,0.0480
rf,Random Forest Regressor,10.5284,166.6384,12.8282,0.4904,0.1875,0.1574,0.1700
lightgbm,Light Gradient Boosting Machine,10.3307,169.3699,12.9167,0.4876,0.1845,0.1547,0.2000
gbr,Gradient Boosting Regressor,11.3271,189.2829,13.6698,0.4205,0.1982,0.1690,0.0620
xgboost,Extreme Gradient Boosting,11.0528,192.9755,13.7998,0.4095,0.2051,0.1664,0.4240
knn,K Neighbors Regressor,11.4621,207.5242,14.3465,0.3652,0.1990,0.1666,0.0380
dt,Decision Tree Regressor,11.6748,222.2443,14.6230,0.3114,0.2104,0.1721,0.0140
dummy,Dummy Regressor,14.4944,345.6971,18.5648,-0.0531,0.2610,0.2206,0.0140
br,Bayesian Ridge,14.7258,353.7026,18.7826,-0.0783,0.2636,0.2244,0.0140


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extra Trees Regressor,8.4338,112.5465,10.6088,0.6456,0.1465,0.1166


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,12.5231,232.5472,15.2495,0.4461,0.2079,0.1812
1,10.5945,226.2167,15.0405,0.2888,0.2610,0.2018
2,9.7241,139.7639,11.8222,0.5135,0.1876,0.1626
3,10.0201,207.2735,14.3970,0.2992,0.1604,0.1165
4,11.6017,196.4407,14.0157,0.4019,0.1954,0.1711
Mean,10.8927,200.4484,14.1050,0.3899,0.2025,0.1667
Std,1.0370,32.9836,1.2239,0.0861,0.0332,0.0283


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extra Trees Regressor,8.4338,112.5465,10.6088,0.6456,0.1465,0.1166


¡Proceso terminado!
ExtraTreesRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región de Arica


La reconfiguración del esquema de validación a 5 folds ha estabilizado exitosamente el modelado para la Región de Arica, seleccionando al Extra Trees Regressor como el estimador óptimo. A diferencia de los ensayos previos plagados de valores negativos, la validación cruzada ahora muestra consistencia estructural absoluta, con todos los folds manteniendo un $R^2$ positivo (rango 0.29 - 0.51), lo que confirma que el modelo ha logrado asimilar los ciclos estacionales sin "puntos ciegos". En el conjunto de prueba final, el algoritmo alcanzó un $R^2$ de 0.65 y un MAPE del 11.66%. Estos indicadores validan al modelo como una herramienta operativamente fiable, capaz de explicar cerca del 65% de la varianza de los siniestros en una región históricamente volátil, manteniendo el error promedio en un margen bajo y controlado.

In [4]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num", "Región de Tarapacá"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Tarapacá", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Tarapacá")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Tarapacá
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
ada,AdaBoost Regressor,25.0862,965.2061,30.7063,0.3771,0.2295,0.1974,0.0800
rf,Random Forest Regressor,25.7792,1000.0671,31.1325,0.3661,0.2329,0.1980,0.1680
et,Extra Trees Regressor,25.9706,1027.6581,31.7122,0.3400,0.2358,0.2021,0.1300
gbr,Gradient Boosting Regressor,26.0399,1017.6487,31.6674,0.3338,0.2311,0.1976,0.0620
knn,K Neighbors Regressor,26.5644,1066.6376,32.1314,0.3303,0.2489,0.2157,0.0380
xgboost,Extreme Gradient Boosting,28.4987,1248.7496,35.2062,0.1594,0.2502,0.2143,0.4400
dt,Decision Tree Regressor,30.1629,1377.6219,36.8359,0.0998,0.2699,0.2276,0.0140
lightgbm,Light Gradient Boosting Machine,30.8766,1629.1118,39.4949,-0.0139,0.2983,0.2564,0.4580
omp,Orthogonal Matching Pursuit,32.1119,1745.6378,41.0483,-0.0787,0.3146,0.2769,0.0160
br,Bayesian Ridge,32.2952,1752.3111,41.0970,-0.0790,0.3150,0.2777,0.0160


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,AdaBoost Regressor,20.1286,684.9464,26.1715,0.1763,0.1819,0.1492


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,30.3271,1422.6243,37.7177,0.3581,0.2214,0.1774
1,20.8406,658.5441,25.6621,0.6285,0.2334,0.1998
2,23.2435,744.6536,27.2883,0.1848,0.1908,0.1716
3,21.4643,706.6028,26.5820,0.4136,0.2056,0.1733
4,24.1583,932.1595,30.5313,0.5044,0.2350,0.2051
Mean,24.0068,892.9169,29.5563,0.4179,0.2173,0.1854
Std,3.3776,280.6262,4.3980,0.1482,0.0169,0.0141


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,AdaBoost Regressor,21.6065,793.0838,28.1617,0.0463,0.1959,0.1617


¡Proceso terminado!
AdaBoostRegressor(random_state=123)
Proceso finalizado para Región de Tarapacá


La implementación de la validación cruzada de 5 folds ha saneado la inestabilidad estructural del modelo, seleccionando al AdaBoost Regressor como el estimador más apto. Los resultados de validación son ahora coherentes y positivos en todos los segmentos temporales, alcanzando un $R^2$ promedio de 0.42 en el entrenamiento, lo que confirma que el modelo logra capturar la tendencia base sin los colapsos matemáticos previos. Sin embargo, se observa una dicotomía en el conjunto de prueba (Test Set): aunque el $R^2$ cae drásticamente a 0.05 (indicando que el modelo no explica bien la varianza de esos meses específicos), el MAPE del 16.17% es sorprendentemente bueno y superior al del entrenamiento. Esto sugiere que el modelo adoptó una estrategia conservadora: ante la alta volatilidad y "ruido" de Tarapacá, aprendió a predecir valores seguros y promedios para minimizar el error porcentual, sacrificando la capacidad de anticipar picos extremos pero garantizando operatividad segura.

In [5]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num", "Región de Antofagasta"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Antofagasta", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Antofagasta")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Antofagasta
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
knn,K Neighbors Regressor,33.8889,1925.2182,43.7811,0.3361,0.1566,0.1242,0.0460
rf,Random Forest Regressor,32.7425,1940.1558,43.7818,0.3133,0.1639,0.1221,0.2080
gbr,Gradient Boosting Regressor,34.3477,2156.4895,45.9196,0.2480,0.1798,0.1310,0.0720
ada,AdaBoost Regressor,34.0703,2225.2748,47.0720,0.2376,0.1814,0.1324,0.0840
et,Extra Trees Regressor,35.2772,2173.3197,46.2971,0.2297,0.1796,0.1358,0.1620
lightgbm,Light Gradient Boosting Machine,36.1270,2268.8003,47.4603,0.2231,0.1760,0.1383,0.5200
xgboost,Extreme Gradient Boosting,35.0781,2232.2403,46.7933,0.1993,0.1843,0.1352,0.4680
omp,Orthogonal Matching Pursuit,38.3497,2681.3267,51.3895,0.1211,0.1844,0.1452,0.0180
br,Bayesian Ridge,38.4951,2703.0109,51.5531,0.1175,0.1854,0.1461,0.0200
huber,Huber Regressor,38.8181,2723.0222,51.7619,0.1047,0.1860,0.1466,0.0260


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,K Neighbors Regressor,25.1385,1017.5508,31.8991,0.7363,0.1324,0.0988


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,27.0595,1625.5417,40.3180,0.4339,0.1218,0.0854
1,32.8393,2099.4903,45.8202,0.3497,0.1978,0.1456
2,36.8929,1991.0893,44.6216,0.6138,0.1882,0.1605
3,32.0750,1584.8016,39.8096,0.1807,0.1339,0.1047
4,30.9812,1973.9539,44.4292,0.2270,0.1405,0.1030
Mean,31.9696,1854.9753,42.9997,0.3610,0.1565,0.1198
Std,3.1652,208.8560,2.4494,0.1548,0.0306,0.0284


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,K Neighbors Regressor,25.3558,1040.8498,32.2622,0.7302,0.1366,0.1014


¡Proceso terminado!
KNeighborsRegressor(n_jobs=-1)
Proceso finalizado para Región de Antofagasta


La aplicación del esquema de 5 folds ha consolidado la estabilidad del proceso, seleccionando al K-Neighbors Regressor (KNN) como el modelo óptimo, un cambio interesante respecto a los modelos de árboles dominantes en otras regiones. La validación cruzada se mantuvo consistente (sin valores negativos), con un $R^2$ promedio de 0.36. Sin embargo, el modelo brilló en el conjunto de prueba (Test Set), alcanzando un $R^2$ de 0.73 y un MAPE del 10.14%. Esto indica que, para Antofagasta, la predicción basada en "vecinos cercanos" (similitud con meses históricos específicos) funciona mejor que tratar de encontrar reglas complejas, logrando una precisión operativa excelente para la toma de decisiones actual.

In [6]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num", "Región de Atacama"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Atacama", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Atacama")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Atacama
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
knn,K Neighbors Regressor,23.1387,879.1542,29.4618,0.1462,0.1786,0.1426,0.0400
ada,AdaBoost Regressor,23.3983,917.2925,29.9052,0.1376,0.1787,0.1455,0.0840
et,Extra Trees Regressor,24.0092,984.5842,30.9774,0.0715,0.1884,0.1494,0.1340
rf,Random Forest Regressor,24.7299,1014.1963,31.5730,0.0305,0.1895,0.1516,0.1780
gbr,Gradient Boosting Regressor,24.9296,1064.5033,32.1510,0.0096,0.1910,0.1541,0.0580
xgboost,Extreme Gradient Boosting,25.8716,1126.8924,32.9831,-0.0415,0.1946,0.1598,0.4000
dummy,Dummy Regressor,25.3545,1141.3938,33.3466,-0.0679,0.1982,0.1597,0.0160
br,Bayesian Ridge,25.5947,1155.1961,33.5969,-0.0885,0.1997,0.1611,0.0220
omp,Orthogonal Matching Pursuit,25.5137,1143.8463,33.5019,-0.0887,0.1993,0.1599,0.0160
huber,Huber Regressor,25.8734,1154.2754,33.6462,-0.0954,0.2001,0.1619,0.0200


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,K Neighbors Regressor,22.7462,859.8723,29.3236,0.3808,0.1606,0.1278


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,18.0714,497.6012,22.3070,0.2231,0.1153,0.0938
1,23.4524,1179.6369,34.3458,0.3092,0.2959,0.2195
2,22.1786,887.0223,29.7829,0.2484,0.1487,0.1148
3,24.4875,969.5641,31.1378,0.0049,0.1767,0.1416
4,19.5625,579.6375,24.0757,0.3241,0.1198,0.0987
Mean,21.5505,822.6924,28.3298,0.2219,0.1713,0.1337
Std,2.3956,252.1424,4.4847,0.1148,0.0661,0.0460


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,K Neighbors Regressor,24.4952,915.9850,30.2652,0.3404,0.1694,0.1402


¡Proceso terminado!
KNeighborsRegressor(n_jobs=-1)
Proceso finalizado para Región de Atacama


La estrategia de validación de 5 folds ha logrado estabilizar matemáticamente el modelado para Atacama, eliminando los coeficientes negativos previos. Se seleccionó nuevamente al K-Neighbors Regressor (KNN) como el estimador óptimo, lo que confirma un patrón en el norte de Chile: ante datos esporádicos y de baja frecuencia, la búsqueda de similitud histórica funciona mejor que las reglas complejas. Si bien el $R^2$ en el conjunto de prueba es modesto (0.34), indicando una correlación estadística débil, el MAPE del 14.02% es bastante competitivo. Esto significa que, aunque el modelo no captura perfectamente la varianza de los picos, su margen de error promedio es bajo y seguro para la operación. La validación cruzada mostró que el Fold 3 fue el punto más débil ($R^2 \approx 0.00$), pero se mantuvo en terreno positivo, validando la robustez estructural del método corregido.

In [7]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num", "Región de Coquimbo"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Coquimbo", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Coquimbo")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Coquimbo
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
huber,Huber Regressor,46.9821,4664.3319,67.0632,0.2381,0.2278,0.1674,0.0300
omp,Orthogonal Matching Pursuit,47.8939,4714.0000,67.5769,0.2293,0.2287,0.1753,0.0200
br,Bayesian Ridge,47.8970,4709.7800,67.6478,0.2270,0.2298,0.1756,0.0200
en,Elastic Net,47.5686,4703.3915,67.6209,0.2264,0.2312,0.1743,0.0220
lasso,Lasso Regression,47.5939,4709.3871,67.6687,0.2251,0.2315,0.1744,0.0180
llar,Lasso Least Angle Regression,47.5939,4709.3871,67.6687,0.2251,0.2315,0.1744,0.0200
ridge,Ridge Regression,47.5983,4710.7990,67.6810,0.2248,0.2316,0.1744,0.0180
lar,Least Angle Regression,47.5990,4710.9854,67.6825,0.2247,0.2316,0.1744,0.0200
lr,Linear Regression,47.5990,4710.9854,67.6825,0.2247,0.2316,0.1744,0.0180
lightgbm,Light Gradient Boosting Machine,50.5141,4725.3086,68.1700,0.2091,0.2347,0.1842,0.2420


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Huber Regressor,58.9853,9876.7869,99.3820,0.1791,0.2879,0.1936


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,48.0639,4493.2900,67.0320,0.1151,0.2144,0.1398
1,51.3663,6779.5132,82.3378,0.0283,0.2910,0.2035
2,39.7655,2731.6300,52.2650,0.4073,0.1943,0.1655
3,56.8561,6503.5546,80.6446,0.1612,0.2758,0.1989
4,38.8317,2817.4830,53.0800,0.4780,0.1640,0.1306
Mean,46.9767,4665.0942,67.0719,0.2380,0.2279,0.1677
Std,6.8760,1733.9073,12.9018,0.1739,0.0483,0.0297


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Huber Regressor,58.9853,9876.7869,99.3820,0.1791,0.2879,0.1936


¡Proceso terminado!
HuberRegressor()
Proceso finalizado para Región de Coquimbo


La reestructuración a 5 folds ha transformado el caos previo en estabilidad, seleccionando al Huber Regressor como el modelo óptimo. Este resultado es revelador: Coquimbo, al ser una zona de transición semiárida con alta variabilidad climática, presenta "ruido" y valores atípicos que confundieron a los modelos complejos (XGBoost y GBR tuvieron rendimientos negativos). El Huber Regressor, diseñado específicamente para ser robusto frente a outliers, logró estabilizar la predicción. Aunque el $R^2$ de prueba es modesto (0.18), indicando que la varianza explicada es baja, el modelo entrega una línea base sólida y operativa con un MAPE del 19.36%. En resumen, el sistema ha priorizado la seguridad y la consistencia sobre la complejidad, entregando una herramienta conservadora que no sobrereacciona a los datos extremos.

In [8]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num", "Región de Valparaíso"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Valparaíso", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Valparaíso")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Valparaíso
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,153.1695,46745.5916,215.0602,0.4865,0.1532,0.1116,0.0660
et,Extra Trees Regressor,155.9336,48820.2610,217.2225,0.4800,0.1563,0.1150,0.1660
xgboost,Extreme Gradient Boosting,160.8513,50402.6077,217.7832,0.4652,0.1542,0.1170,0.6200
rf,Random Forest Regressor,160.3344,48721.5082,219.1125,0.4645,0.1562,0.1173,0.2080
ada,AdaBoost Regressor,180.9404,56919.8170,237.4416,0.3644,0.1721,0.1363,0.0860
dt,Decision Tree Regressor,191.4238,66660.9895,255.7859,0.2783,0.1797,0.1378,0.0180
lightgbm,Light Gradient Boosting Machine,198.2555,67180.4088,257.8919,0.2541,0.1862,0.1478,0.2140
knn,K Neighbors Regressor,210.4589,74941.4550,272.1009,0.1682,0.1948,0.1525,0.0460
omp,Orthogonal Matching Pursuit,221.1837,87853.5493,295.7296,0.0091,0.2137,0.1686,0.0180
br,Bayesian Ridge,221.6429,88243.1923,296.4605,0.0065,0.2144,0.1690,0.0200


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,144.4632,36239.2782,190.3662,0.7154,0.1322,0.1043


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,188.3296,55696.0933,236.0002,0.5349,0.1683,0.1385
1,147.8858,51473.6667,226.8781,0.3398,0.2039,0.1386
2,130.7160,23610.7977,153.6581,0.6509,0.1160,0.0996
3,156.1889,45303.2615,212.8456,0.4376,0.1462,0.1044
4,163.1894,55636.2722,235.8734,0.5031,0.1369,0.1010
Mean,157.2620,46344.0183,213.0511,0.4932,0.1542,0.1164
Std,18.9425,11983.3474,30.8748,0.1033,0.0299,0.0181


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,149.5055,36000.6305,189.7383,0.7173,0.1274,0.1039


¡Proceso terminado!
GradientBoostingRegressor(random_state=123)
Proceso finalizado para Región de Valparaíso


La reingeniería de la estrategia de validación a 5 folds resultó crítica para la Región de Valparaíso, logrando estabilizar la zona con mayor volatilidad siniestral del estudio y seleccionando al Gradient Boosting Regressor como el estimador óptimo. A diferencia de los ensayos anteriores, el modelo eliminó los quiebres estructurales de invierno, manteniendo una consistencia positiva en todas las iteraciones de validación cruzada. En el conjunto de prueba final, el algoritmo demostró una capacidad predictiva superior, alcanzando un $R^2$ de 0.72 y un notable MAPE del 10.39%, lo que implica un margen de error operativo de apenas el 10%. Este resultado valida que la ampliación de la ventana temporal permitió al modelo internalizar correctamente la compleja estacionalidad de la interfaz urbano-forestal, entregando por fin una herramienta robusta para la gestión de riesgos en la zona central.

In [9]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num", "Región Metropolitana"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región Metropolitana", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región Metropolitana")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región Metropolitana
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,272.5921,158673.3904,385.8618,0.7501,0.1270,0.0898,0.0820
xgboost,Extreme Gradient Boosting,297.7760,179309.5627,411.0194,0.7162,0.1349,0.0984,0.4720
rf,Random Forest Regressor,298.1367,182379.3775,408.1683,0.7148,0.1328,0.0986,0.2060
et,Extra Trees Regressor,311.4953,185179.6230,411.9824,0.7095,0.1342,0.1018,0.1560
ada,AdaBoost Regressor,325.9399,222682.9520,456.8164,0.6472,0.1530,0.1118,0.0640
knn,K Neighbors Regressor,373.1785,259778.7029,496.3800,0.5943,0.1642,0.1253,0.0420
dt,Decision Tree Regressor,396.1105,266024.1790,509.0804,0.5669,0.1640,0.1253,0.0200
lightgbm,Light Gradient Boosting Machine,409.0785,304788.6497,546.3067,0.5190,0.1844,0.1431,0.1820
omp,Orthogonal Matching Pursuit,455.9093,369945.3947,599.1410,0.4042,0.1999,0.1615,0.0160
br,Bayesian Ridge,461.0982,377488.0074,605.7846,0.3905,0.2023,0.1634,0.0180


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,212.9740,73292.8636,270.7265,0.9407,0.1012,0.0753


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,372.0731,193292.6061,439.6506,0.7894,0.1368,0.1130
1,301.8933,279262.3507,528.4528,0.5765,0.2295,0.1414
2,170.4084,50515.5013,224.7565,0.9160,0.0808,0.0642
3,246.5611,95143.8144,308.4539,0.7872,0.0841,0.0689
4,358.7967,193576.7140,439.9735,0.6699,0.1155,0.0959
Mean,289.9465,162358.1973,388.2575,0.7478,0.1293,0.0967
Std,74.5647,80777.0715,107.7698,0.1157,0.0542,0.0286


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,212.9740,73292.8636,270.7265,0.9407,0.1012,0.0753


¡Proceso terminado!
GradientBoostingRegressor(random_state=123)
Proceso finalizado para Región Metropolitana


La implementación de la validación cruzada de 5 folds ha perfeccionado la capacidad predictiva para la Región Metropolitana, seleccionando nuevamente al Gradient Boosting Regressor como el algoritmo dominante. La estabilidad del modelo es absoluta: el promedio de validación ($R^2$ 0.75) confirma la eliminación de los "puntos ciegos" estacionales. No obstante, lo más impresionante es su desempeño en el conjunto de prueba, donde alcanzó un $R^2$ de 0.94 y un MAPE de apenas 7.53%. Estos indicadores son los más altos de todo el estudio nacional, sugiriendo que la siniestralidad en la capital obedece a patrones estacionales y antropogénicos muy marcados que el modelo ha logrado decodificar con una precisión casi determinística, convirtiéndolo en una herramienta de gestión de riesgo de primer nivel.

In [10]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región del Libertador Bernardo O'higgins"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región del Libertador Bernardo O'higgins", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región del Libertador Bernardo O'higgins")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región del Libertador Bernardo O'higgins
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,201.1366,82362.1287,275.5411,0.3554,0.2154,0.1651,0.2060
et,Extra Trees Regressor,207.4287,87714.0360,284.0554,0.3218,0.2227,0.1710,0.1460
gbr,Gradient Boosting Regressor,214.8023,86022.3950,282.9494,0.3169,0.2232,0.1733,0.0760
ada,AdaBoost Regressor,221.5039,93455.1623,299.3013,0.1962,0.2357,0.1881,0.0620
knn,K Neighbors Regressor,231.5704,102730.9190,308.6574,0.1815,0.2411,0.1871,0.0400
lightgbm,Light Gradient Boosting Machine,225.1825,100853.1684,306.8360,0.1435,0.2399,0.1902,0.1800
xgboost,Extreme Gradient Boosting,236.9194,107223.7315,320.1916,0.0830,0.2537,0.1981,0.4340
huber,Huber Regressor,255.4244,115663.0565,331.1233,0.0170,0.2609,0.2155,0.0240
en,Elastic Net,257.0969,118485.1038,335.4057,-0.0263,0.2641,0.2206,0.0180
lar,Least Angle Regression,257.1625,118689.6947,335.5917,-0.0264,0.2643,0.2205,0.0160


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,196.6946,57514.1433,239.8211,0.5424,0.1978,0.1695


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,266.4449,165747.9396,407.1215,0.3597,0.2549,0.1814
1,163.3143,59236.5414,243.3856,0.1752,0.2594,0.1884
2,154.6532,36673.5266,191.5033,0.4293,0.1958,0.1638
3,173.5850,44077.5609,209.9466,0.4865,0.1629,0.1325
4,208.6241,85080.0548,291.6849,0.4637,0.2013,0.1571
Mean,193.3243,78163.1247,268.7284,0.3829,0.2149,0.1646
Std,40.9058,46819.5935,77.1245,0.1123,0.0370,0.0197


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,200.3512,55894.5069,236.4202,0.5553,0.1918,0.1730


¡Proceso terminado!
RandomForestRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región del Libertador Bernardo O'higgins


La estrategia de validación corregida a 5 folds ha logrado estabilizar el modelado para la Región de O'Higgins, eliminando las inconsistencias negativas previas y seleccionando al Random Forest Regressor como el estimador óptimo. Aunque esta región presenta una dinámica transicional compleja (interfaz agrícola-forestal), la validación cruzada se mantuvo robusta con todos los coeficientes positivos ($R^2$ entre 0.18 y 0.49). En el conjunto de prueba final, el modelo alcanzó un $R^2$ de 0.56 y un MAPE del 17.30%. Si bien la precisión es menor que en la Metropolitana o Valparaíso (debido probablemente al ruido introducido por quemas agrícolas controladas que se mezclan con siniestros), el modelo es ahora estructuralmente sano y ofrece una línea base confiable para la gestión operativa, sin los colapsos matemáticos observados anteriormente.

In [11]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región del Maule"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región del Maule", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región del Maule")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región del Maule
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,115.1711,26498.6766,153.9565,0.5617,0.1954,0.1493,0.2080
gbr,Gradient Boosting Regressor,120.8637,26797.4947,156.2908,0.5548,0.1986,0.1551,0.0700
xgboost,Extreme Gradient Boosting,123.7107,29401.9275,161.2391,0.5484,0.2081,0.1593,0.4660
et,Extra Trees Regressor,130.5594,33232.9004,176.2401,0.4729,0.2135,0.1641,0.1580
ada,AdaBoost Regressor,133.8691,32265.7490,174.4502,0.4586,0.2233,0.1766,0.0800
dt,Decision Tree Regressor,135.8790,36453.3238,179.9449,0.3632,0.2240,0.1709,0.0200
lightgbm,Light Gradient Boosting Machine,148.1532,39442.7685,193.5530,0.3437,0.2380,0.1883,0.2000
en,Elastic Net,197.8487,60465.1301,242.6626,-0.1664,0.3075,0.2593,0.0200
br,Bayesian Ridge,199.8169,61451.8519,244.7848,-0.1756,0.3093,0.2613,0.0180
llar,Lasso Least Angle Regression,197.6042,60804.2455,243.0456,-0.1799,0.3092,0.2589,0.0200


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,158.7273,68424.5105,261.5808,0.5121,0.2334,0.1711


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,166.4727,45955.1674,214.3716,0.4788,0.2466,0.1918
1,115.5804,22431.0229,149.7699,0.5913,0.2358,0.1779
2,94.2520,14161.1229,119.0005,0.2050,0.1547,0.1297
3,88.5121,14620.2867,120.9144,0.8203,0.1165,0.0894
4,163.6808,46840.5292,216.4267,0.4644,0.2395,0.2089
Mean,125.6996,28801.6258,164.0966,0.5120,0.1986,0.1595
Std,33.4044,14667.5513,43.2889,0.1995,0.0530,0.0439


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,158.7273,68424.5105,261.5808,0.5121,0.2334,0.1711


¡Proceso terminado!
RandomForestRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región del Maule


La estrategia de validación de 5 folds ha corregido exitosamente la inestabilidad en la Región del Maule, consolidando al Random Forest Regressor como el modelo más robusto frente al colapso de los estimadores lineales. La validación cruzada muestra ahora una salud estructural impecable, eliminando los valores negativos y oscilando entre un ajuste moderado ($R^2$ 0.20) y uno excelente ($R^2$ 0.82) dependiendo de la severidad de la temporada. En el conjunto de prueba, se obtuvo un $R^2$ de 0.51 y un MAPE del 17.11%. Si bien el error es mayor que en la zona central, esto es esperable dada la naturaleza de los "megaincendios" forestales del Maule; el modelo ha dejado de fallar matemáticamente y ahora entrega una línea base operativa confiable que captura la compleja no-linealidad de la carga de combustible forestal de la zona.

In [12]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región del Ñuble"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región del Ñuble", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región del Ñuble")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región del Ñuble
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,79.6095,12789.7937,105.2318,0.6381,0.1976,0.1594,0.1620
rf,Random Forest Regressor,81.3638,13203.8121,107.0084,0.6109,0.2038,0.1658,0.2140
gbr,Gradient Boosting Regressor,87.1660,14047.5325,111.8512,0.5576,0.2134,0.1777,0.0760
xgboost,Extreme Gradient Boosting,90.8614,15356.1427,116.9851,0.5162,0.2222,0.1817,0.4900
ada,AdaBoost Regressor,90.0485,15512.4874,119.8906,0.5049,0.2346,0.1951,0.0740
lightgbm,Light Gradient Boosting Machine,91.1246,16191.2258,122.1326,0.5023,0.2290,0.1865,0.5100
huber,Huber Regressor,103.4785,20721.1615,140.0030,0.3436,0.2695,0.2157,0.0380
en,Elastic Net,105.0011,20703.2088,140.0778,0.3281,0.2733,0.2252,0.0140
llar,Lasso Least Angle Regression,105.1418,20729.0754,140.2264,0.3257,0.2747,0.2255,0.0180
ridge,Ridge Regression,105.1396,20728.2538,140.2255,0.3257,0.2748,0.2254,0.0180


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extra Trees Regressor,79.1465,19099.2836,138.2002,0.5811,0.2160,0.1393


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,74.5876,10343.3591,101.7023,0.7336,0.1974,0.1568
1,79.2022,14365.1258,119.8546,0.3770,0.3091,0.2255
2,69.4869,6296.6234,79.3513,0.5150,0.1883,0.1795
3,72.3254,7729.7197,87.9188,0.7472,0.1580,0.1311
4,134.9567,35272.2597,187.8091,0.4155,0.2743,0.2296
Mean,86.1118,14801.4175,115.3272,0.5576,0.2254,0.1845
Std,24.6286,10596.7962,38.7434,0.1559,0.0568,0.0384


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extra Trees Regressor,79.1465,19099.2836,138.2002,0.5811,0.2160,0.1393


¡Proceso terminado!
ExtraTreesRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región del Ñuble


La validación corregida a 5 folds ha resuelto la inestabilidad estructural en la Región del Ñuble, seleccionando al Extra Trees Regressor como el modelo superior. A diferencia de los árboles estándar, este algoritmo añade aleatoriedad en los cortes, lo que resultó crucial para filtrar el "ruido" de una zona con alta actividad mixta (agrícola y forestal). La validación cruzada es ahora totalmente consistente, eliminando los colapsos previos y oscilando en rangos positivos ($R^2$ entre 0.37 y 0.74). En el conjunto de prueba, el modelo alcanzó un $R^2$ de 0.58 y un MAPE del 13.93%. Este margen de error inferior al 14% valida al modelo como una herramienta operativamente segura, capaz de distinguir eficazmente entre la estacionalidad normal y los picos de siniestralidad en una de las regiones más activas del centro-sur.

In [13]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región del Bío Bío"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región del Bío Bío", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región del Bío Bío")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región del Bío Bío
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,151.8119,46618.3578,207.7578,0.5002,0.1603,0.1226,0.2020
gbr,Gradient Boosting Regressor,159.3131,50128.2190,215.2158,0.4791,0.1607,0.1261,0.0740
et,Extra Trees Regressor,153.8801,48730.8678,215.1280,0.4565,0.1660,0.1255,0.1560
xgboost,Extreme Gradient Boosting,168.5282,54648.4410,229.1353,0.3557,0.1740,0.1352,0.5240
lightgbm,Light Gradient Boosting Machine,174.3175,59729.4687,238.8429,0.3256,0.1851,0.1415,0.6640
ada,AdaBoost Regressor,185.2069,61837.0561,242.3289,0.2761,0.1878,0.1528,0.1020
huber,Huber Regressor,202.6562,76642.5442,270.2805,0.1064,0.2155,0.1626,0.0300
en,Elastic Net,204.3576,76749.3891,271.5407,0.0802,0.2161,0.1673,0.0120
llar,Lasso Least Angle Regression,203.1385,76952.0268,271.7416,0.0752,0.2172,0.1664,0.0240
ridge,Ridge Regression,203.1103,76948.9846,271.7321,0.0752,0.2172,0.1664,0.0100


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,158.5123,62885.9589,250.7707,0.4834,0.1740,0.1212


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,158.8475,46127.9460,214.7742,0.5662,0.1570,0.1200
1,125.8008,33734.9338,183.6707,0.4969,0.1858,0.1237
2,128.2576,28376.6933,168.4538,0.1617,0.1572,0.1297
3,134.3684,26443.0456,162.6132,0.7413,0.1199,0.1011
4,217.8046,99255.0358,315.0477,0.4416,0.1889,0.1432
Mean,153.0158,46787.5309,208.9119,0.4815,0.1617,0.1235
Std,34.4469,27116.4210,56.0655,0.1891,0.0249,0.0137


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,158.5123,62885.9589,250.7707,0.4834,0.1740,0.1212


¡Proceso terminado!
RandomForestRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región del Bío Bío


La corrección de la estrategia de validación a 5 folds ha saneado exitosamente la modelación en la Región del Bío Bío, históricamente la zona con mayor carga de combustible forestal del país. Se seleccionó al Random Forest Regressor como el estimador óptimo, demostrando una estabilidad estructural que los modelos anteriores no lograban: se eliminaron totalmente los coeficientes negativos, oscilando la validación cruzada entre un ajuste moderado ($R^2$ 0.16) y uno alto ($R^2$ 0.74). En el conjunto de prueba final, el modelo alcanzó un $R^2$ de 0.48 y un MAPE del 12.12%. Este margen de error de apenas el 12% representa un éxito operativo significativo, dado que predecir siniestros en el epicentro de la industria forestal implica manejar una varianza extrema que el modelo ha logrado normalizar eficazmente.

In [14]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región de la Araucanía"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de la Araucanía", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de la Araucanía")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de la Araucanía
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,161.8669,56809.4110,231.8656,0.6383,0.1816,0.1377,0.1700
et,Extra Trees Regressor,172.0809,61420.6364,241.9111,0.6036,0.1860,0.1458,0.1320
gbr,Gradient Boosting Regressor,169.1869,60851.3749,242.0842,0.5997,0.1902,0.1442,0.0560
xgboost,Extreme Gradient Boosting,183.0594,71052.0118,260.4040,0.5463,0.2015,0.1531,0.4360
lightgbm,Light Gradient Boosting Machine,195.0473,79466.0299,270.6769,0.5350,0.2219,0.1720,0.1920
ada,AdaBoost Regressor,206.7391,78894.3694,277.3060,0.4573,0.2231,0.1868,0.0660
dt,Decision Tree Regressor,202.9610,87381.5867,291.4361,0.3983,0.2299,0.1706,0.0160
huber,Huber Regressor,248.8596,121543.1061,339.6728,0.2150,0.2756,0.2132,0.0200
en,Elastic Net,252.5522,117412.0995,336.9254,0.2011,0.2822,0.2292,0.0160
br,Bayesian Ridge,253.1255,118029.2019,337.6498,0.1984,0.2827,0.2295,0.0180


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,141.5331,33546.2157,183.1563,0.7739,0.1635,0.1306


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,146.1512,34621.6431,186.0689,0.7585,0.1677,0.1430
1,151.1270,43338.1854,208.1783,0.6352,0.2138,0.1520
2,149.5855,46309.5811,215.1966,0.4190,0.2017,0.1653
3,204.3728,98007.1945,313.0610,0.6586,0.1910,0.1372
4,248.9162,97610.7697,312.4272,0.5134,0.2228,0.1987
Mean,180.0305,63977.4748,246.9864,0.5970,0.1994,0.1592
Std,40.6150,27889.4960,54.5453,0.1183,0.0192,0.0219


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,141.5331,33546.2157,183.1563,0.7739,0.1635,0.1306


¡Proceso terminado!
RandomForestRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región de la Araucanía


La reestructuración de la validación a 5 folds ha sido un éxito rotundo en la Región de la Araucanía, la "zona roja" forestal del país, seleccionando al Random Forest Regressor como el estimador más robusto. La estabilidad estructural se recuperó por completo: los folds que antes colapsaban ahora muestran un rendimiento consistente y positivo (rango $R^2$ 0.41 - 0.76), demostrando que el modelo ha logrado internalizar la compleja estacionalidad y conflictividad de la zona. En el conjunto de prueba final, el desempeño fue sobresaliente con un $R^2$ de 0.77 y un MAPE del 13.06%. Este resultado es crítico desde el punto de vista operativo, ya que lograr una precisión del 87% (error del 13%) en la región con mayor carga de combustible y variabilidad de eventos valida al modelo como una herramienta de gestión de alto nivel.

In [15]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región de los Ríos"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de los Ríos", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de los Ríos")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de los Ríos
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,52.8763,5207.0892,71.1426,0.6982,0.1936,0.1483,0.2000
gbr,Gradient Boosting Regressor,56.2117,5528.7199,73.5770,0.6766,0.2002,0.1553,0.0960
dt,Decision Tree Regressor,62.1652,6745.0167,79.5661,0.6145,0.2010,0.1620,0.0200
xgboost,Extreme Gradient Boosting,61.8822,6491.0774,79.0184,0.6138,0.2137,0.1751,0.7960
et,Extra Trees Regressor,56.7595,6078.8301,77.4742,0.6078,0.2059,0.1604,0.1300
ada,AdaBoost Regressor,62.2232,6891.2350,82.6535,0.5574,0.2267,0.1856,0.0960
lightgbm,Light Gradient Boosting Machine,66.2181,8363.1615,89.6165,0.5219,0.2330,0.1824,0.1980
knn,K Neighbors Regressor,73.4095,10093.7268,98.1944,0.3883,0.2578,0.2011,0.0480
huber,Huber Regressor,77.7368,11639.5568,104.8063,0.3345,0.2739,0.2119,0.0220
en,Elastic Net,80.2408,11507.9744,105.3050,0.3299,0.2736,0.2246,0.0180


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,38.6542,2538.1511,50.3801,0.8211,0.1358,0.1041


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,61.3411,6284.7209,79.2762,0.6560,0.1977,0.1691
1,49.6916,6374.9887,79.8435,0.5954,0.2543,0.1662
2,48.9236,5061.3134,71.1429,0.4785,0.2299,0.1873
3,63.7372,7578.4456,87.0543,0.6702,0.1972,0.1541
4,64.0200,7664.2664,87.5458,0.6238,0.1935,0.1312
Mean,57.5427,6592.7470,80.9725,0.6048,0.2145,0.1616
Std,6.7924,959.9316,6.0161,0.0682,0.0238,0.0185


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,38.6542,2538.1511,50.3801,0.8211,0.1358,0.1041


¡Proceso terminado!
RandomForestRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región de los Ríos


La corrección de la estrategia a 5 folds ha sido sumamente efectiva para la Región de Los Ríos, neutralizando por completo la inestabilidad que anteriormente afectaba a los modelos del sur (el antiguo problema del "Fold 8"). Se seleccionó al Random Forest Regressor como el estimador óptimo, exhibiendo una solidez estructural notable: la validación cruzada osciló de manera segura entre un $R^2$ de 0.47 y 0.67, sin ningún coeficiente negativo. El desempeño en el conjunto de prueba (Test Set) fue sobresaliente, alcanzando un $R^2$ de 0.82 y un MAPE del 10.41%. Este resultado es de primer nivel operativo; lograr un error cercano al 10% en una región caracterizada por su densa selva valdiviana y geografía compleja indica que el modelo ha capturado perfectamente la estacionalidad y los factores de riesgo climáticos.

In [16]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región de los Lagos"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de los Lagos", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de los Lagos")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de los Lagos
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,115.3349,23262.3897,150.5534,0.0646,0.1433,0.1124,0.1580
rf,Random Forest Regressor,114.6628,23623.0890,152.7370,-0.0067,0.1464,0.1128,0.2300
ada,AdaBoost Regressor,116.0092,24334.6333,154.9326,-0.0167,0.1486,0.1156,0.0980
knn,K Neighbors Regressor,118.0649,25776.7237,158.4922,-0.0467,0.1519,0.1159,0.0460
dummy,Dummy Regressor,121.6196,28481.9933,165.0463,-0.0733,0.1629,0.1229,0.0180
huber,Huber Regressor,117.6790,26671.5574,161.6554,-0.0796,0.1592,0.1180,0.0340
lightgbm,Light Gradient Boosting Machine,117.7806,27668.6715,162.7185,-0.0820,0.1614,0.1176,0.4700
br,Bayesian Ridge,121.9423,28892.6001,166.2701,-0.0889,0.1642,0.1236,0.0220
gbr,Gradient Boosting Regressor,119.8798,25115.4475,157.6573,-0.0912,0.1494,0.1171,0.0860
en,Elastic Net,117.2924,27023.7471,162.9159,-0.1060,0.1610,0.1184,0.0180


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extra Trees Regressor,90.3785,15561.7867,124.7469,0.4809,0.1153,0.0865


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,144.2041,37830.5581,194.5008,0.0371,0.1708,0.1288
1,96.5343,21886.3785,147.9405,0.1950,0.1814,0.1156
2,100.4835,15626.2796,125.0051,0.0863,0.1306,0.1085
3,115.4203,28559.9762,168.9970,0.2828,0.1479,0.1065
4,94.9305,11994.2579,109.5183,-0.0465,0.1042,0.0930
Mean,110.3145,23179.4901,149.1923,0.1109,0.1470,0.1105
Std,18.4271,9245.9285,30.3503,0.1161,0.0277,0.0117


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extra Trees Regressor,102.7554,17796.4987,133.4035,0.4064,0.1293,0.1018


¡Proceso terminado!
ExtraTreesRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región de los Lagos


La estrategia de validación de 5 folds ha logrado "rescatar" el modelado de la Región de Los Lagos, pasando de un escenario de nula capacidad predictiva ($R^2 \approx 0.01$ en intentos previos) a una operatividad moderada. Se seleccionó al Extra Trees Regressor como el estimador óptimo, cuya aleatoriedad en la selección de cortes resultó crucial para manejar el alto "ruido estocástico" de una zona climáticamente inestable y lluviosa. Aunque la validación cruzada sigue mostrando desafíos (con un promedio de $R^2$ de 0.11), el desempeño en el conjunto de prueba (Test Set) fue notablemente superior, alcanzando un $R^2$ de 0.41 y un MAPE del 10.18%. Esto indica que, a pesar de la dificultad para explicar la varianza total (debido a la irregularidad de los eventos en la zona), el modelo logra estimar la magnitud de los siniestros con un error porcentual muy bajo, validándose como una herramienta conservadora y segura.

In [17]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región de Aysén del General Carlos Ibáñez del Campo"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Aysén del General Carlos Ibáñez del Campo", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Aysén del General Carlos Ibáñez del Campo")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Aysén del General Carlos Ibáñez del Campo
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,14.8910,364.1244,18.6632,0.1981,0.2484,0.2101,0.1480
ada,AdaBoost Regressor,15.1696,358.7971,18.6627,0.1949,0.2367,0.2101,0.0320
rf,Random Forest Regressor,15.4389,382.1670,19.1067,0.1405,0.2497,0.2145,0.1920
gbr,Gradient Boosting Regressor,15.8018,392.3186,19.3600,0.1218,0.2457,0.2131,0.0640
lightgbm,Light Gradient Boosting Machine,15.5685,408.3382,19.9782,0.0682,0.2797,0.2383,0.2580
xgboost,Extreme Gradient Boosting,16.8149,462.0273,20.8432,-0.0119,0.2624,0.2278,0.7980
dummy,Dummy Regressor,17.0190,467.9532,21.3302,-0.0411,0.2964,0.2640,0.0140
br,Bayesian Ridge,16.9763,471.3380,21.4114,-0.0492,0.2980,0.2653,0.0180
omp,Orthogonal Matching Pursuit,17.1579,484.0515,21.7251,-0.0831,0.3021,0.2692,0.0160
knn,K Neighbors Regressor,17.4820,476.6706,21.6236,-0.0884,0.2996,0.2682,0.0420


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extra Trees Regressor,14.3254,325.8542,18.0514,0.5158,0.2353,0.2021


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,15.6674,329.1349,18.1421,0.2258,0.2057,0.1836
1,20.3286,568.6959,23.8473,0.1522,0.3594,0.3478
2,15.7295,406.1278,20.1526,0.2624,0.3620,0.3160
3,12.3603,311.6683,17.6541,0.1559,0.2450,0.1764
4,10.6440,190.6158,13.8064,0.1981,0.1448,0.1111
Mean,14.9460,361.2486,18.7205,0.1989,0.2634,0.2270
Std,3.3257,124.6223,3.2850,0.0419,0.0856,0.0899


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extra Trees Regressor,16.5940,454.2851,21.3140,0.3250,0.3018,0.2604


¡Proceso terminado!
ExtraTreesRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región de Aysén del General Carlos Ibáñez del Campo


La validación de 5 folds ha rescatado exitosamente el modelado para Aysén. Se seleccionó al Extra Trees Regressor como el estimador óptimo, una elección lógica para datos escasos y ruidosos. A diferencia de los modelos anteriores, la validación cruzada es ahora totalmente estable (todos los folds positivos, rango 0.15 - 0.26), eliminando los colapsos invernales.En el conjunto de prueba, el modelo alcanzó un $R^2$ de 0.33 y un MAPE del 26.04%. Aunque estas métricas son más bajas que en la zona central, representan un éxito técnico significativo para Aysén. Esta región tiene muy pocos siniestros (datos "dispersos"), lo que hace que cualquier error pequeño infle el porcentaje (MAPE). Que el modelo logre explicar un tercio de la varianza ($R^2$ 0.33) en una zona de eventos tan esporádicos es un resultado operativo válido y conservador.

In [18]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región de Magallanes y de la Antártica Chilena"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Magallanes y de la Antártica Chilena", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Magallanes y de la Antártica Chilena")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Magallanes y de la Antártica Chilena
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,17.1197,483.9061,21.4715,0.1294,0.2078,0.1731,0.1760
omp,Orthogonal Matching Pursuit,16.7096,498.6237,21.9492,0.0851,0.2185,0.1767,0.0220
br,Bayesian Ridge,16.8783,503.7462,22.0825,0.0722,0.2200,0.1786,0.0200
huber,Huber Regressor,16.7650,523.3915,22.3567,0.0569,0.2208,0.1735,0.0280
lasso,Lasso Regression,17.1009,524.1679,22.4432,0.0464,0.2231,0.1799,0.0240
llar,Lasso Least Angle Regression,17.1009,524.1679,22.4432,0.0464,0.2231,0.1799,0.0220
en,Elastic Net,17.1099,524.8591,22.4576,0.0452,0.2232,0.1800,0.0280
knn,K Neighbors Regressor,18.4857,513.9240,22.3491,0.0402,0.2168,0.1886,0.0500
lr,Linear Regression,17.1518,528.1656,22.5221,0.0400,0.2238,0.1803,0.0220
ridge,Ridge Regression,17.1513,528.1314,22.5215,0.0400,0.2238,0.1803,0.0300


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,12.7885,239.2203,15.4667,0.2541,0.1775,0.1454


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,16.1833,370.2581,19.2421,0.1412,0.2056,0.1797
1,20.3156,750.2400,27.3905,0.0153,0.2509,0.2032
2,13.7177,315.8187,17.7713,-0.0502,0.2129,0.1789
3,18.1077,474.2551,21.7774,0.1214,0.2054,0.1769
4,19.1084,634.4091,25.1875,0.0081,0.2296,0.1797
Mean,17.4865,508.9962,22.2738,0.0471,0.2209,0.1837
Std,2.3205,162.2958,3.5883,0.0726,0.0174,0.0098


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,12.7885,239.2203,15.4667,0.2541,0.1775,0.1454


¡Proceso terminado!
LGBMRegressor(device='gpu', n_jobs=-1, random_state=123)
Proceso finalizado para Región de Magallanes y de la Antártica Chilena


El ajuste de la validación a 5 folds logró rescatar la modelación para la Región de Magallanes, superando la inviabilidad estadística previa y seleccionando a LightGBM como el estimador más eficaz. Aunque la escasez de eventos en la zona austral limita la capacidad de explicar la varianza total ($R^2$ de validación bajo pero positivo), el desempeño en el conjunto de prueba fue técnicamente exitoso, logrando un $R^2$ de 0.25 y un MAPE del 14.54%. Esto valida al modelo como una herramienta conservadora y robusta: en lugar de sobreajustarse al ruido o colapsar en invierno, el algoritmo aprendió a filtrar las falsas alarmas, entregando una línea base de riesgo con un margen de error operativo acotado y seguro para una zona de eventos esporádicos.